# Real-Time AI Agent Interfaces with AG-UI on Amazon Bedrock AgentCore

| Information | Details |
|:------------|:--------|
| Agent Framework | Strands Agents |
| LLM | Anthropic Claude Sonnet 4 |
| Components | AgentCore Runtime, AG-UI event streaming, S3/CloudFront frontend |
| CLI | AgentCore CLI (`@aws/agentcore`) |

This notebook deploys a Strands agent to AgentCore that emits real AG-UI events — tool calls, reasoning, text streaming — viewable in a browser frontend.

**Prerequisites**: SageMaker Studio, AWS credentials with Bedrock access, Node.js 20+, supporting infrastructure (Lambda proxy + S3/CloudFront) already deployed.

In [ ]:
%pip install --force-reinstall -U typing_extensions strands-agents strands-agents-tools strands-agents-bedrock boto3 uv --quiet
!npm install -g @aws/agentcore --silent 2>/dev/null && echo "AgentCore CLI installed: $(agentcore --version)"
!sudo apt-get install -y zip --quiet 2>/dev/null || echo "zip already installed"
!which uv && which zip

## Step 1: Write the AG-UI agent server

FastAPI server with Strands agent + 7 tools. Emits real AG-UI events by inspecting the Strands message history after each invocation. AgentCore runs this on `/invocations` (SSE) and `/ping` (health).

In [ ]:
%%writefile agui_agent.py
"""AG-UI Agent Server - real AG-UI events from Strands agent execution."""
import asyncio, json, uuid, time, random, os
from datetime import datetime, timedelta
from typing import Optional
from fastapi import FastAPI, Request
from fastapi.responses import StreamingResponse, JSONResponse
from strands import Agent, tool
from strands.models import BedrockModel

@tool
def search_knowledge_base(query: str, category: str = "all") -> dict:
    """Search the knowledge base for information on a topic.
    Args:
        query: The search query
        category: Category - all, technical, business, research
    """
    db = {
        "technical": [{"title": "Microservices Patterns", "snippet": "Best practices for distributed systems.", "relevance": 0.95},
                      {"title": "Event-Driven Design", "snippet": "Event sourcing and CQRS for scalability.", "relevance": 0.91},
                      {"title": "Container Orchestration", "snippet": "K8s and ECS strategies.", "relevance": 0.88}],
        "business": [{"title": "Digital Transformation ROI", "snippet": "Measuring cloud migration returns.", "relevance": 0.93},
                     {"title": "Cost Optimization", "snippet": "Reducing cloud spend 30-40%.", "relevance": 0.90}],
        "research": [{"title": "LLM Agent Architectures", "snippet": "ReAct, CoT, tool-use patterns.", "relevance": 0.97},
                     {"title": "RAG Patterns", "snippet": "Grounding LLMs with knowledge bases.", "relevance": 0.94}],
    }
    results = [] if category == "all" else db.get(category, db["technical"])
    if category == "all":
        for v in db.values(): results.extend(v)
        random.shuffle(results); results = results[:5]
    for r in results: r["relevance"] = round(random.uniform(0.75, 0.99), 2); r["title"] = f"{r['title']}: {query}"
    return {"query": query, "category": category, "results": results, "total_found": len(results)}

@tool
def calculator(expression: str) -> dict:
    """Perform mathematical calculations.
    Args:
        expression: Math expression to evaluate
    """
    allowed = {"abs": abs, "round": round, "min": min, "max": max, "pow": pow}
    try: return {"expression": expression, "result": eval(expression, {"__builtins__": {}}, allowed)}
    except Exception as e: return {"expression": expression, "error": str(e)}

@tool
def get_inventory(source: str, status_filter: str = "all") -> dict:
    """Retrieve inventory data from a warehouse or store.
    Args:
        source: Warehouse or store name
        status_filter: Filter - all, active, low_stock, out_of_stock
    """
    inv = [{"sku": "SKU-1001", "name": "Wireless Headphones", "status": "active", "quantity": 342, "price": 79.99, "reorder_point": 100},
           {"sku": "SKU-1002", "name": "USB-C Hub", "status": "active", "quantity": 156, "price": 49.99, "reorder_point": 50},
           {"sku": "SKU-1003", "name": "Mechanical Keyboard", "status": "low_stock", "quantity": 23, "price": 129.99, "reorder_point": 50},
           {"sku": "SKU-1004", "name": "4K Monitor", "status": "active", "quantity": 89, "price": 399.99, "reorder_point": 30},
           {"sku": "SKU-1005", "name": "Webcam HD", "status": "out_of_stock", "quantity": 0, "price": 59.99, "reorder_point": 75},
           {"sku": "SKU-1006", "name": "Laptop Stand", "status": "low_stock", "quantity": 12, "price": 34.99, "reorder_point": 40},
           {"sku": "SKU-1007", "name": "Wireless Mouse", "status": "active", "quantity": 501, "price": 29.99, "reorder_point": 100},
           {"sku": "SKU-1008", "name": "Noise Cancelling Earbuds", "status": "active", "quantity": 210, "price": 149.99, "reorder_point": 80}]
    if status_filter != "all": inv = [i for i in inv if i["status"] == status_filter]
    return {"source": source, "filter": status_filter, "items": inv, "count": len(inv),
            "total_inventory_value": round(sum(i["quantity"]*i["price"] for i in inv), 2), "retrieved_at": datetime.now().isoformat()}

@tool
def analyze_trends(data_type: str, period: str = "7d") -> dict:
    """Analyze trends in sales or errors.
    Args:
        data_type: Type - sales, errors
        period: Time period - 24h, 7d, 30d
    """
    n = {"24h": 24, "7d": 7, "30d": 30}.get(period, 7)
    if data_type == "sales":
        pts = [{"date": (datetime.now()-timedelta(days=i)).strftime("%Y-%m-%d"), "value": round(15000+random.uniform(-3000,5000),2)} for i in range(n)]
        return {"data_type": "sales", "period": period, "data_points": pts[:5], "trend": "increasing" if pts[0]["value"]>pts[-1]["value"] else "decreasing",
                "average": round(sum(d["value"] for d in pts)/len(pts),2)}
    elif data_type == "errors":
        pts = [{"date": (datetime.now()-timedelta(days=i)).strftime("%Y-%m-%d"), "count": random.randint(2,45), "severity": random.choice(["low","medium","high"])} for i in range(n)]
        return {"data_type": "errors", "period": period, "data_points": pts[:5], "total_errors": sum(d["count"] for d in pts)}
    return {"error": "Unknown type", "available": ["sales", "errors"]}

@tool
def lookup_customer(customer_id: str) -> dict:
    """Look up customer information.
    Args:
        customer_id: Customer ID (CUST-001, CUST-002, CUST-003)
    """
    db = {"CUST-001": {"name": "Acme Corp", "tier": "Enterprise", "annual_spend": 125000, "satisfaction_score": 4.8, "open_tickets": 1},
          "CUST-002": {"name": "TechStart Inc", "tier": "Growth", "annual_spend": 45000, "satisfaction_score": 4.2, "open_tickets": 3},
          "CUST-003": {"name": "Global Retail Co", "tier": "Enterprise", "annual_spend": 280000, "satisfaction_score": 4.9, "open_tickets": 0}}
    return db.get(customer_id, {"error": "Not found", "available": list(db.keys())})

@tool
def generate_report(title: str, report_type: str = "summary") -> dict:
    """Generate a business report.
    Args:
        title: Report title
        report_type: Type - summary, executive, action_items
    """
    return {"title": title, "report_type": report_type, "generated_at": datetime.now().isoformat(),
            "recommendations": [{"priority": "high", "action": "Restock low inventory", "impact": "Prevent $12K lost sales"},
                                {"priority": "medium", "action": "Review open tickets", "impact": "Maintain satisfaction > 4.5"}], "status": "generated"}

@tool
def send_alert(channel: str, message: str, severity: str = "info") -> dict:
    """Send an alert notification.
    Args:
        channel: Channel - email, slack, pagerduty
        message: Alert message
        severity: Severity - info, warning, critical
    """
    return {"alert_id": f"ALERT-{random.randint(10000,99999)}", "channel": channel, "severity": severity,
            "message": message, "status": "sent", "sent_at": datetime.now().isoformat()}

REGION = os.environ.get('AWS_DEFAULT_REGION', os.environ.get('AWS_REGION', 'us-west-2'))

def create_agent():
    model = BedrockModel(model_id='us.anthropic.claude-sonnet-4-20250514-v1:0', region_name=REGION)
    return Agent(model=model, tools=[search_knowledge_base, calculator, get_inventory, analyze_trends, lookup_customer, generate_report, send_alert],
                 system_prompt='You are a business operations assistant. Use multiple tools when needed. Always look up data before recommending. Calculate numbers, dont estimate. If you find issues, send alerts. Be concise.')

def agui_event(t, **d):
    return f'data: {json.dumps({"type": t, "eventId": str(uuid.uuid4()), "timestamp": time.time(), **d})}\n\n'

def extract_tool_events(messages):
    events = []
    for msg in messages:
        for block in msg.get('content', []):
            if not isinstance(block, dict): continue
            if 'toolUse' in block:
                tu = block['toolUse']
                tc_id = tu.get('toolUseId', str(uuid.uuid4()))
                events.append(('TOOL_CALL_START', {'toolCallId': tc_id, 'toolName': tu.get('name','')}))
                args_str = json.dumps(tu.get('input', {}))
                for i in range(0, len(args_str), 50):
                    events.append(('TOOL_CALL_ARGS', {'toolCallId': tc_id, 'delta': args_str[i:i+50]}))
                events.append(('TOOL_CALL_END', {'toolCallId': tc_id}))
            elif 'toolResult' in block:
                tr = block['toolResult']
                tc_id = tr.get('toolUseId', str(uuid.uuid4()))
                content = tr.get('content', [])
                txt = content[0]['text'] if isinstance(content, list) and content and isinstance(content[0], dict) else str(content)
                try: parsed = json.loads(txt)
                except: parsed = {'result': txt}
                events.append(('TOOL_CALL_RESULT', {'toolCallId': tc_id, 'result': parsed}))
    return events

app = FastAPI()

@app.get('/ping')
async def ping(): return JSONResponse({'status': 'healthy'})

@app.post('/invocations')
async def invocations(request: Request):
    body = await request.json()
    prompt = body.get('prompt', body.get('message', ''))
    run_id = str(uuid.uuid4())
    async def stream():
        agent = create_agent()
        yield agui_event('RUN_STARTED', runId=run_id)
        yield agui_event('STATE_SNAPSHOT', snapshot={'status': 'processing', 'tools_invoked': []})
        rid = str(uuid.uuid4())
        yield agui_event('REASONING_START', reasoningId=rid)
        yield agui_event('REASONING_MESSAGE', reasoningId=rid, delta='Analyzing request and planning tool usage...')
        yield agui_event('REASONING_END', reasoningId=rid)
        try:
            result = await asyncio.get_event_loop().run_in_executor(None, lambda: agent(prompt))
            tool_events = extract_tool_events(agent.messages)
            tools_used = [e[1]['toolName'] for e in tool_events if e[0] == 'TOOL_CALL_START']
            for et, ed in tool_events: yield agui_event(et, **ed)
            if tools_used: yield agui_event('STATE_DELTA', delta={'tools_invoked': tools_used, 'status': 'responding'})
            msg_id = str(uuid.uuid4())
            yield agui_event('TEXT_MESSAGE_START', messageId=msg_id, role='assistant')
            for i, w in enumerate(str(result).split(' ')):
                yield agui_event('TEXT_MESSAGE_CONTENT', messageId=msg_id, delta=(w if i==0 else f' {w}'))
                await asyncio.sleep(0.02)
            yield agui_event('TEXT_MESSAGE_END', messageId=msg_id)
            yield agui_event('STATE_DELTA', delta={'status': 'idle'})
            yield agui_event('RUN_FINISHED', runId=run_id)
        except Exception as e:
            yield agui_event('RUN_ERROR', runId=run_id, error={'code': 'AGENT_ERROR', 'message': str(e)})
    return StreamingResponse(stream(), media_type='text/event-stream', headers={'Cache-Control': 'no-cache', 'Connection': 'keep-alive'})

if __name__ == '__main__':
    import uvicorn
    # NOTE: 0.0.0.0 binds to all interfaces — required for AgentCore container routing.
    # For local development, use '127.0.0.1' to restrict to localhost.
    uvicorn.run(app, host='0.0.0.0', port=int(os.environ.get('PORT', '8080')))

In [ ]:
%%writefile requirements.txt
strands-agents
strands-agents-tools
strands-agents-bedrock
fastapi
uvicorn[standard]

## Step 2: Test the agent locally

In [ ]:
from strands import Agent, tool
from strands.models import BedrockModel
from boto3.session import Session

@tool
def test_calc(expression: str) -> dict:
    """Calculate math.
    Args:
        expression: Math expression
    """
    return {"result": eval(expression, {"__builtins__": {}})}

region = Session().region_name
model = BedrockModel(model_id='us.anthropic.claude-sonnet-4-20250514-v1:0', region_name=region)
test_agent = Agent(model=model, tools=[test_calc], system_prompt='Always use the calculator tool for math.')
result = test_agent('What is 42 * 17?')
print(result)

## Step 3: Deploy to AgentCore Runtime

`agentcore create` scaffolds the project and generates `.bedrock_agentcore.yaml`, then `agentcore deploy` packages and deploys to AgentCore.

In [ ]:
import shutil, os, yaml
from boto3.session import Session

NOTEBOOK_DIR = os.getcwd()
PROJECT_DIR = os.path.join(NOTEBOOK_DIR, 'aguisampleagent')

# Clean up previous project folder if it exists
if os.path.isdir(PROJECT_DIR):
    shutil.rmtree(PROJECT_DIR)
    print('Removed existing aguisampleagent/')

!agentcore create --project-name aguisampleagent --agent-framework Strands --model-provider Bedrock --memory NO_MEMORY --non-interactive --no-venv

In [ ]:
import os, shutil, yaml
from boto3.session import Session

region = Session().region_name

# cd into project dir
os.chdir(PROJECT_DIR)

# Copy our agent code into the src/ directory (where the config expects it)
shutil.copy2(os.path.join(NOTEBOOK_DIR, 'agui_agent.py'), 'src/main.py')
shutil.copy2(os.path.join(NOTEBOOK_DIR, 'requirements.txt'), 'src/requirements.txt')

# Update config: set region and runtime version
config_path = '.bedrock_agentcore.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

agent_key = list(config['agents'].keys())[0]
config['agents'][agent_key]['runtime_type'] = 'PYTHON_3_12'
config['agents'][agent_key]['aws']['region'] = region

with open(config_path, 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print(f"Entrypoint: {config['agents'][agent_key]['entrypoint']}")
print(f"Source path: {config['agents'][agent_key]['source_path']}")
print(f"Region: {region}")
!agentcore deploy --auto-update-on-conflict

In [ ]:
!agentcore status

## Step 4: Test on AgentCore

In [ ]:
!agentcore invoke "Check warehouse-east for low stock items and send a slack alert"

In [ ]:
import boto3, json, subprocess, os
from boto3.session import Session

region = Session().region_name
account_id = Session().client('sts').get_caller_identity()['Account']

# Read agent ARN from the deployed config
import yaml
with open('.bedrock_agentcore.yaml') as f:
    config = yaml.safe_load(f)

agent_key = list(config['agents'].keys())[0]
agent_cfg = config['agents'][agent_key]['bedrock_agentcore']
agent_arn = agent_cfg.get('agent_arn')
agent_id = agent_cfg.get('agent_id')

if not agent_arn and agent_id:
    agent_arn = f"arn:aws:bedrock-agentcore:{region}:{account_id}:runtime/{agent_id}"

print(f"Agent ID: {agent_id}")
print(f"Agent ARN: {agent_arn}")

# Invoke via boto3 for raw SSE inspection
client = boto3.client('bedrock-agentcore', region_name=region)

resp = client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier='DEFAULT',
    payload=json.dumps({'prompt': 'Look up customer CUST-001 and CUST-003, compare their annual spend, generate a report'})
)

print(f'\nContent-Type: {resp.get("contentType", "unknown")}\n')
ct = resp.get('contentType', '')
if 'text/event-stream' in ct:
    for line in resp['response'].iter_lines(chunk_size=1):
        if line:
            d = line.decode('utf-8')
            if d.startswith('data: '):
                ev = json.loads(d[6:])
                extra = {k:v for k,v in ev.items() if k not in ('type','eventId','timestamp')}
                print(f"[{ev['type']}] {json.dumps(extra) if extra else ''}")
else:
    for chunk in resp.get('response', []):
        print(chunk.decode('utf-8') if isinstance(chunk, bytes) else chunk)

## Step 5: Wire up the frontend

Update the Lambda proxy with the agent ARN and print the frontend URL.

In [ ]:
import boto3, subprocess, json, yaml
from boto3.session import Session

region = Session().region_name
account_id = Session().client('sts').get_caller_identity()['Account']

# Read agent ARN from the deployed config
with open('.bedrock_agentcore.yaml') as f:
    config = yaml.safe_load(f)

agent_key = list(config['agents'].keys())[0]
agent_cfg = config['agents'][agent_key]['bedrock_agentcore']
agent_arn = agent_cfg.get('agent_arn')
agent_id = agent_cfg.get('agent_id')

if not agent_arn and agent_id:
    agent_arn = f"arn:aws:bedrock-agentcore:{region}:{account_id}:runtime/{agent_id}"

if agent_arn:
    lambda_client = boto3.client('lambda', region_name=region)
    try:
        lambda_client.update_function_configuration(
            FunctionName='agui-bedrock-agentcore-proxy',
            Environment={'Variables': {'AGENT_RUNTIME_ARN': agent_arn, 'REGION': region}}
        )
        print(f'Lambda proxy updated: {agent_arn}')
    except Exception as e:
        print(f'Lambda update error: {e}')
else:
    print('Could not determine agent ARN. Run agentcore status to check deployment.')

# Get frontend URL
result = subprocess.run(
    ['aws', 'cloudformation', 'describe-stacks', '--stack-name', 'agui-bedrock-agentcore', '--region', region, '--output', 'json'],
    capture_output=True, text=True
)
if result.returncode == 0:
    outputs = {o['OutputKey']: o['OutputValue'] for o in json.loads(result.stdout)['Stacks'][0]['Outputs']}
    print(f'\n{"="*50}')
    print(f'AG-UI FRONTEND: {outputs.get("FrontendUrl", "N/A")}')
    print(f'{"="*50}')
    print(f'\nTry these queries:')
    print(f'  1. Check warehouse-east for low stock items, calculate restock cost, send a slack alert')
    print(f'  2. Look up customer CUST-001 and CUST-003, compare their spend')
    print(f'  3. Analyze error trends for 7 days, analyze sales for 30 days, generate a report')
else:
    print('CloudFormation stack not found.')

## Cleanup

In [ ]:
# Uncomment to clean up

# # Delete AgentCore deployment
# !agentcore destroy

# # Delete supporting infrastructure
# import boto3
# from boto3.session import Session
# region = Session().region_name
# os.chdir(NOTEBOOK_DIR)
# !aws s3 rm s3://$(aws cloudformation describe-stacks --stack-name agui-bedrock-agentcore --region {region} --query 'Stacks[0].Outputs[?OutputKey==`FrontendBucketName`].OutputValue' --output text)/ --recursive
# !aws cloudformation delete-stack --stack-name agui-bedrock-agentcore --region {region}

# # Clean up local files
# !rm -rf aguisampleagent/ agui_agent.py requirements.txt
# print('Done')